In [8]:
!pip install keras-tuner


   ---------------------------------------- 0/2 [kt-legacy]
   ---------------------------------------- 0/2 [kt-legacy]
   ---------------------------------------- 0/2 [kt-legacy]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   ---------------------------------------- 2/2 [keras-tuner]



In [25]:
import keras
from keras import layers
#import keras_tuner as kt
from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, accuracy_score

from tensorflow import keras
from keras import layers
#import keras_tuner as kt

from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd


In [26]:
#import keras_tuner as kt
import keras_tuner as kt

In [27]:
import keras_tuner

https://keras.io/keras_tuner/getting_started/

In [28]:
hp = kt.HyperParameters()
# Load dataset
def load_dataset():
    data = fetch_california_housing()
    X, y = data.data, data.target

    return X, y

In [29]:
load_dataset()

(array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]]),
 array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894]))

In [30]:
# Preprocessing
scaler = StandardScaler()

def preprocess_data(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test

In [31]:
# Build Keras model with tunable hyperparameters
def build_model(hp, input_shape):

    # Define Hyperparams

    # Tune the activation function to use.
    activation=hp.Choice("activation", ["relu", "tanh"])
    # The number of hidden layers
    num_layers = hp.Int('num_layers', 2, 4)
    # Define the optimizer learning rate as a hyperparameter.
    learning_rate = hp.Float("lr", min_value=1e-4, max_value=1e-2, sampling="log")


    # Build the model
    model = keras.Sequential()
    # Input Layer
    model.add(layers.Input(shape=(input_shape,)))

    # Hidden Layers
    for i in range(num_layers):
        # Tune number of units.
        units=hp.Int(f'units_{i}', 32, 128, step=32)
        #print(units)
        model.add(layers.Dense(units=units,activation=activation))
        # Dropout - Tune whether to use dropout.
        if hp.Boolean("dropout"):
            model.add(layers.Dropout(rate=0.25))


    # Output Layer
    model.add(layers.Dense(1))

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='mse', metrics=['mae'])

    return model

In [32]:
# Run the pipeline
def run_pipeline():
    print('Loading data.......')
    X, y = load_dataset()
    X_train, X_test, y_train, y_test = preprocess_data(X, y)

    print('Tuning parameters.......')

    tuner = kt.RandomSearch(
        lambda hp: build_model(hp, X_train.shape[1]),
        objective='val_loss',
        max_trials=5,
        executions_per_trial=2,
        #directory='./',
        #project_name='regression_model'
    )

    tuner.search(X_train, y_train, epochs=20, validation_split=0.2, verbose=2)
    best_model = tuner.get_best_models(num_models=1)[0]
    best_hp = tuner.get_best_hyperparameters(1)[0]

    print('Training the model ...............')
    best_model.fit(X_train, y_train, epochs=30, validation_split=0.2, verbose=2)
    y_pred = best_model.predict(X_test).flatten()
    #build_model(hp=hp,input_shape=X_train.shape[1])
    return best_model

In [33]:
# Run the pipeline
print("=== REGRESSION: California Housing ===")
model = run_pipeline()

=== REGRESSION: California Housing ===
Loading data.......
Tuning parameters.......
Reloading Tuner from .\untitled_project\tuner0.json

Search: Running Trial #3

Value             |Best Value So Far |Hyperparameter
tanh              |tanh              |activation
4                 |4                 |num_layers
0.00084854        |0.00084854        |lr
96                |96                |units_0
True              |True              |dropout
32                |32                |units_1



Traceback (most recent call last):
  File "C:\Users\Dell\anaconda3\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dell\anaconda3\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
  File "C:\Users\Dell\anaconda3\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 309, in run_trial
    self._configure_tensorboard_dir(callbacks, trial, execution)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dell\anaconda3\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 421, in _configure_tensorboard_dir
    from tensorboard.plugins.hparams import api as hparams_api
ModuleNotFoundError: No module named 'tensorboard'


RuntimeError: Number of consecutive failures exceeded the limit of 3.
Traceback (most recent call last):
  File "C:\Users\Dell\anaconda3\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dell\anaconda3\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
  File "C:\Users\Dell\anaconda3\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 309, in run_trial
    self._configure_tensorboard_dir(callbacks, trial, execution)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dell\anaconda3\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 421, in _configure_tensorboard_dir
    from tensorboard.plugins.hparams import api as hparams_api
ModuleNotFoundError: No module n

In [34]:
# Predict for unseen data
new_data = np.array([[8.3252, 41.0, 6.9841, 1.0238, 322.0, 2.5556, 37.88, -122.23]])
# California Housing
new_data_scaled = scaler.transform(new_data)
prediction = model.predict(new_data_scaled)
prediction

NameError: name 'model' is not defined

In [35]:
#### Classification

In [36]:
def load_dataset():
    data = pd.read_csv('../../../datasets/pima_indians_diabetes.csv')
    X= data.iloc[:,:8]
    y= data.iloc[:,8]
    return X, y

In [37]:
# Preprocessing
def preprocess_data(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test

In [38]:
# Build Keras model with tunable hyperparameters
def build_model(hp, input_shape):
    # Hyperparameters
    num_layers = hp.Int('num_layers', 1, 3)
    activation = hp.Choice('activation', ['relu', 'tanh'])
    learning_rate = hp.Choice('lr', [1e-2, 1e-3])

    # Build the network
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_shape,)))


    for i in range(num_layers):
        units = hp.Int(f'units_{i}', 32, 128, step=32)
        model.add(layers.Dense(units=units,
                               activation=activation))
    model.add(layers.Dense(1, activation='sigmoid'))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [39]:
# Run the pipeline
def run_pipeline():
    print('Loading Data.......')
    X, y = load_dataset()
    X_train, X_test, y_train, y_test = preprocess_data(X, y)
    print('Tuning parameters.......')
    tuner = kt.RandomSearch(
        lambda hp: build_model(hp, X_train.shape[1]),
        objective='val_loss',
        max_trials=5,
        executions_per_trial=2,
        #directory='./',
        #project_name='classification_model'
    )

    tuner.search(X_train, y_train, epochs=20, validation_split=0.2, verbose=2)
    best_model = tuner.get_best_models(num_models=1)[0]
    best_hp = tuner.get_best_hyperparameters(1)[0]

    print(f"\n🔧 Best Hyperparameters for Classification:")
    print(best_hp.values)

    print('Training the model')
    # best_model = build_model(best_hp, X_train.shape[1], task, num_classes)
    best_model.fit(X_train, y_train, epochs=30, validation_split=0.2, verbose=2)
    y_pred = best_model.predict(X_test)

    y_pred_class = (y_pred > 0.5).astype(int)
    print("✅ Final Accuracy:", accuracy_score(y_test, y_pred_class))
    return best_model

In [40]:
print("\n=== CLASSIFICATION: Diabetes (Binary) ===")
model = run_pipeline()


=== CLASSIFICATION: Diabetes (Binary) ===
Loading Data.......


FileNotFoundError: [Errno 2] No such file or directory: '../../../datasets/pima_indians_diabetes.csv'